## Creating Embeddings

In [0]:
df = spark.read.csv("dbfs:/product_details/products_AT_de_AT.csv", sep=";", header=True, inferSchema=True)
df.createOrReplaceTempView('product_details_at')

df = df.filter(df.Langbeschreibung.isNotNull())
display(df)

In [0]:
df = spark.sql("""
        select p.*, i.item_id, i.category_caption_level_2, i.family 
        from product_details_at p
        inner join curated.dim_item_detail_pim i 
        on p.identifier = i.identifier
        """)

In [0]:
from pyspark.sql.functions import regexp_replace, concat_ws, initcap, expr

df = df.withColumn("Name AX", initcap(df["Name AX"]))
df = df.withColumnRenamed("Name AX", "Name_ax")

df = df.withColumn("Inhaltsstoffe_new",expr("array_join(regexp_extract_all(Inhaltsstoffe, '<Description>(.*?)</Description>'), ', ')"))

df = df.withColumn("Warnungen_new", regexp_replace(df.Warnungen, r'<CautionId>.*?</CautionId>', ''))
df = df.withColumn("Warnungen_new", regexp_replace(df.Warnungen_new, r'<[^>]*>', ''))

df = df.withColumnRenamed("URL-Prefix", "url_prefix")
df = df.withColumn("category", concat_ws(" ", df["url_prefix"], df['category_caption_level_2'], df['family']))


df = df.withColumn("Langbeschreibung_new", regexp_replace(df.Langbeschreibung, r'<[^>]*>', ''))
df = df.withColumn("descriptions", concat_ws(" ", df["Langbeschreibung_new"], df['Keywords'], df['ABDA Warengruppe Code'], df['seo_slug']))
df = df.withColumn("descriptions", regexp_replace(df.descriptions, r'\s{3,}', '  '))
display(df.limit(10))

# Format 1

In [0]:
import threading
from pyspark.sql.functions import pandas_udf, col
from pyspark.sql.types import ArrayType, FloatType
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

In [0]:
def get_model():
    # Cache the model per executor to avoid reloading
    if not hasattr(get_model, "model"):
        from sentence_transformers import SentenceTransformer
        get_model.model = SentenceTransformer('distiluse-base-multilingual-cased-v2')
    return get_model.model

In [0]:
@pandas_udf(ArrayType(FloatType()))
def compute_embedding_udf(desc: pd.Series) -> pd.Series:
    # Use the single 'descriptiosn' column and handle missing values
    texts = desc.fillna("").astype(str)
    
    # Get the cached model
    model = get_model()
    
    # Convert the pandas Series to a list for batch processing
    text_list = texts.tolist()
    
    # --- Parallel processing with ThreadPoolExecutor ---
    # Tune num_threads based on the cores available per executor
    num_threads = 8  # adjust this as needed based on available cores per executor
    chunk_size = len(text_list) // num_threads + 1
    chunks = [text_list[i:i + chunk_size] for i in range(0, len(text_list), chunk_size)]
    
    with ThreadPoolExecutor(max_workers=num_threads) as executor:
        # Each thread encodes its chunk concurrently
        results = list(executor.map(lambda chunk: model.encode(chunk, batch_size=128, show_progress_bar=False), chunks))
    
    # Flatten the list of results (order is preserved with executor.map)
    embeddings = [embedding for sublist in results for embedding in sublist]
    
    # Convert each embedding to a list
    embeddings_list = [emb.tolist() for emb in embeddings]
    return pd.Series(embeddings_list)

In [0]:
df = df.repartition(48)

In [0]:
df_with_embeddings = df.withColumn(
    "embedding",
    compute_embedding_udf(col("descriptions"))
).cache()

In [0]:
from pyspark.sql.functions import col

df_with_embeddings = df_with_embeddings.select(
    [col(column).alias(column.replace(" ", "_").replace("(", "").replace(")", "")) for column in df_with_embeddings.columns]
)

df_with_embeddings = df_with_embeddings.select(['identifier', 'Name_Shop', 'url_prefix', 'Kurzbeschreibung', 'seo_slug', 'Langbeschreibung_new', 'Inhaltsstoffe_new', 'Warnungen_new', 'descriptions', 'embedding']).withColumnRenamed('URL-Prefix', 'url_prefix')

In [0]:
df_with_embeddings.write.format("delta").mode("overwrite").saveAsTable("datascience.at_product_details_embeddings")

## Similarity search

In [0]:
%sql
select * from datascience.at_product_details_embeddings
limit 10

In [0]:
df = spark.sql("""
               select a.*, p.family,p.category_caption_level_1,p.category_caption_level_2,p.category_caption_level_3,p.item_id,p.pzn
               from datascience.at_product_details_embeddings a
               inner join (select * from curated.dim_item_detail_pim where sub_company_id = 'Shop-AT') p
               on a.identifier = p.identifier
               """)

In [0]:
%sql
select * from curated.dim_item_detail_pim
limit 10

In [0]:
pip install faiss-cpu

In [0]:
import faiss
import numpy as np
import pandas as pd

In [0]:
pdf = df.toPandas()

In [0]:
embeddings =  np.vstack(pdf['embedding'].values).astype(np.float32)
d = embeddings.shape[1]  # dimension of each embedding
d

In [0]:
id_mapping = pdf['identifier'].tolist()

In [0]:
index_flat = faiss.IndexFlatL2(d)
index_flat.add(embeddings)

In [0]:
#Inverted File (IVF) index – approximate search.
nlist = 100  # Number of clusters; adjust as needed.
quantizer = faiss.IndexFlatL2(d)  # Quantizer for clustering embeddings.
index_ivf = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_L2)
index_ivf.train(embeddings)  # Must train the index before adding vectors.
index_ivf.add(embeddings)


In [0]:
#HNSW index (Hierarchical Navigable Small World graphs) for efficient approximate search.
index_hnsw = faiss.IndexHNSWFlat(d, 32)  # 32 is a typical value for the parameter M.
index_hnsw.add(embeddings)


In [0]:
#Product Quantization (PQ) index – compresses vectors for faster search.
m = 8  # Number of subquantizers (tune this parameter)
index_pq = faiss.IndexPQ(d, m, 8)  # 8 bits per sub-vector.
index_pq.train(embeddings)
index_pq.add(embeddings)


In [0]:
def search_index(index, query_embeddings, k=5):
    # 'k' should be higher than the desired number of similar identifiers to account for self-match.
    distances, indices = index.search(query_embeddings, k)
    return distances, indices

In [0]:
k = 10

In [0]:
indexes = {
    'Flat (Exact)': index_flat,
    'IVF (Approximate)': index_ivf,
    'HNSW (Approximate)': index_hnsw,
    'PQ (Approximate)': index_pq
}

In [0]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Computing a global PCA transformation on all embeddings so that the 2D coordinates are comparable.
pca = PCA(n_components=2)
global_embeddings_2d = pca.fit_transform(embeddings)

In [0]:
def get_recommendations_and_plot(query_identifier, k=6):
    """
    Given a query identifier, this function finds the top recommendations (excluding self-match)
    for all four indexes and plots the query and recommended embeddings in 4 subplots.
    
    Parameters:
      query_identifier (str): Identifier to query.
      k (int): Number of nearest neighbors to search for (typically k-1 recommendations after filtering self-match).
    """
    if query_identifier not in id_mapping:
        print(f"Identifier '{query_identifier}' not found!")
        return
    
    # Get the row index and corresponding embedding of the query.
    query_idx = id_mapping.index(query_identifier)
    query_embedding = embeddings[query_idx].reshape(1, -1)
    query_coord = global_embeddings_2d[query_idx]  # 2D coordinates from PCA
    
    # Setup a 2x2 grid for subplots.
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    axes = axes.flatten()  # easier iteration over subplots
    
    # For each index, get recommendations and plot.
    for ax, (index_name, index) in zip(axes, indexes.items()):
        # Perform a k-NN search on the query embedding.
        distances, indices = index.search(query_embedding, k)
        indices = indices.flatten().tolist()
        # Exclude the query itself (often the closest match) and get up to (k-1) recommendations.
        rec_indices = [idx for idx in indices if idx != query_idx][:k-1]
        
        # Get the 2D PCA coordinates for the recommendations.
        rec_coords = global_embeddings_2d[rec_indices]
        
        # Plot the query point (red) and recommended points (blue).
        ax.scatter(query_coord[0], query_coord[1], color='red', s=100, label='Query')
        ax.scatter(rec_coords[:, 0], rec_coords[:, 1], color='blue', s=80, label='Recommendations')
        
        # Annotate the query point.
        ax.annotate(id_mapping[query_idx], (query_coord[0], query_coord[1]), 
                    textcoords="offset points", xytext=(5,5), color='red')
        # Annotate each recommended point.
        for idx, (x, y) in zip(rec_indices, rec_coords):
            ax.annotate(id_mapping[idx], (x, y), textcoords="offset points", xytext=(5,5), color='blue')
        
        ax.set_title(index_name)
        ax.legend()
    
    plt.tight_layout()
    plt.show()
    
    # Also print the recommendations for each index.
    for index_name, index in indexes.items():
        distances, indices = index.search(query_embedding, k)
        indices = indices.flatten().tolist()
        rec_indices = [idx for idx in indices if idx != query_idx][:k-1]
        rec_ids = [id_mapping[idx] for idx in rec_indices]
        print(f"{index_name} recommendations for '{query_identifier}': {rec_ids}")


In [0]:
def get_recommendations_and_plot_all(query_identifier, k=6):
    """
    Given a query identifier, find the top recommendations for all four indexes,
    and plot the full embeddings with transparency, highlighting the query and
    recommendations in each subplot.
    
    Parameters:
      query_identifier (str): Identifier to query.
      k (int): Number of nearest neighbors to search for (use k-1 recommendations after filtering self-match).
    """
    if query_identifier not in id_mapping:
        print(f"Identifier '{query_identifier}' not found!")
        return
    
    # Get the row index and corresponding embedding of the query.
    query_idx = id_mapping.index(query_identifier)
    query_embedding = embeddings[query_idx].reshape(1, -1)
    query_coord = global_embeddings_2d[query_idx]
    
    # Setup a 2x2 grid for subplots.
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    axes = axes.flatten()  # for easier iteration
    
    # For each index, get recommendations and plot.
    for ax, (index_name, index) in zip(axes, indexes.items()):
        # Plot all embeddings in the background with transparency.
        ax.scatter(global_embeddings_2d[:, 0], global_embeddings_2d[:, 1],
                   color='gray', alpha=0.2, s=10, label='All Embeddings')
        
        # Perform k-NN search on the query embedding.
        distances, indices = index.search(query_embedding, k)
        indices = indices.flatten().tolist()
        rec_indices = [idx for idx in indices if idx != query_idx][:k-1]
        
        # Get the 2D PCA coordinates for recommendations.
        rec_coords = global_embeddings_2d[rec_indices]
        
        # Plot query and recommendations.
        ax.scatter(query_coord[0], query_coord[1], color='red', s=100, label='Query')
        ax.scatter(rec_coords[:, 0], rec_coords[:, 1], color='blue', s=80, label='Recommendations')
        
        # Annotate the query.
        ax.annotate(id_mapping[query_idx], (query_coord[0], query_coord[1]),
                    textcoords="offset points", xytext=(5,5), color='red')
        # Annotate each recommended point.
        for idx, (x, y) in zip(rec_indices, rec_coords):
            ax.annotate(id_mapping[idx], (x, y),
                        textcoords="offset points", xytext=(5,5), color='blue')
        
        ax.set_title(index_name)
        ax.legend()
    
    plt.tight_layout()
    plt.show()
    
    # Also print the recommendations for each index.
    for index_name, index in indexes.items():
        distances, indices = index.search(query_embedding, k)
        indices = indices.flatten().tolist()
        rec_indices = [idx for idx in indices if idx != query_idx][:k-1]
        rec_ids = [id_mapping[idx] for idx in rec_indices]
        print(f"{index_name} recommendations for '{query_identifier}': {rec_ids}")

In [0]:
get_recommendations_and_plot("00000425")

In [0]:
get_recommendations_and_plot_all("00000425")

In [0]:
%sql
select * from datascience.at_product_details_embeddings
where identifier in ('00000425', '00000170', '00000431', '01771745', '02893686', '02894183')

In [0]:
%sql
select * from datascience.at_product_details_embeddings
where identifier in ('00000425', '04210438', '08479249', '07246738', '07595019', '07455695')

from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, random_state=42)
global_embeddings_2d = tsne.fit_transform(embeddings)

def get_recommendations_and_plot_tsne(query_identifier, k=6):
    """
    Given a query identifier, find the top recommendations for all four indexes,
    and plot all embeddings in the background (with transparency), highlighting the
    query and recommendations.
    
    Parameters:
      query_identifier (str): Identifier to query.
      k (int): Number of nearest neighbors to search for (retrieve k-1 recommendations after filtering self-match).
    """
    if query_identifier not in id_mapping:
        print(f"Identifier '{query_identifier}' not found!")
        return
    
    # Get the row index and corresponding embedding of the query.
    query_idx = id_mapping.index(query_identifier)
    query_embedding = embeddings[query_idx].reshape(1, -1)
    query_coord = global_embeddings_2d[query_idx]
    
    # Setup a 2x2 grid for subplots.
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    axes = axes.flatten()  # for easier iteration
    
    # For each index, get recommendations and plot.
    for ax, (index_name, index) in zip(axes, indexes.items()):
        # Plot all embeddings in the background with transparency.
        ax.scatter(global_embeddings_2d[:, 0], global_embeddings_2d[:, 1],
                   color='gray', alpha=0.2, s=10, label='All Embeddings')
        
        # Perform a k-NN search on the query embedding.
        distances, indices = index.search(query_embedding, k)
        indices = indices.flatten().tolist()
        # Exclude the query itself and take up to (k-1) recommendations.
        rec_indices = [idx for idx in indices if idx != query_idx][:k-1]
        
        # Get the 2D t-SNE coordinates for the recommendations.
        rec_coords = global_embeddings_2d[rec_indices]
        
        # Plot query and recommendations.
        ax.scatter(query_coord[0], query_coord[1], color='red', s=100, label='Query')
        ax.scatter(rec_coords[:, 0], rec_coords[:, 1], color='blue', s=80, label='Recommendations')
        
        # Annotate the query.
        ax.annotate(id_mapping[query_idx], (query_coord[0], query_coord[1]),
                    textcoords="offset points", xytext=(5,5), color='red')
        # Annotate each recommended point.
        for idx, (x, y) in zip(rec_indices, rec_coords):
            ax.annotate(id_mapping[idx], (x, y),
                        textcoords="offset points", xytext=(5,5), color='blue')
        
        ax.set_title(index_name)
        ax.legend()
    
    plt.tight_layout()
    plt.show()
    
    # Also print the recommendations for each index.
    for index_name, index in indexes.items():
        distances, indices = index.search(query_embedding, k)
        indices = indices.flatten().tolist()
        rec_indices = [idx for idx in indices if idx != query_idx][:k-1]
        rec_ids = [id_mapping[idx] for idx in rec_indices]
        print(f"{index_name} recommendations for '{query_identifier}': {rec_ids}")


# Format 2

In [0]:
import threading
from pyspark.sql.functions import pandas_udf, col
from pyspark.sql.types import (
    StructType, StructField, ArrayType, FloatType
)
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

In [0]:
def get_model():
    # Cache the model per executor to avoid reloading
    if not hasattr(get_model, "model"):
        from sentence_transformers import SentenceTransformer
        get_model.model = SentenceTransformer('distiluse-base-multilingual-cased-v2')
    return get_model.model

In [0]:

# Define output schema: one field per input column with an "_embedding" suffix
embedding_schema = StructType([
    StructField("Name_AX_embedding", ArrayType(FloatType())),
    StructField("Inhaltsstoffe_new_embedding", ArrayType(FloatType())),
    StructField("Warnungen_new_embedding", ArrayType(FloatType())),
    StructField("category_embedding", ArrayType(FloatType())),
    StructField("descriptions_embedding", ArrayType(FloatType()))
])


In [0]:
@pandas_udf(embedding_schema)
def compute_all_embeddings_udf(name_ax: pd.Series,
                               inhaltsstoffe: pd.Series,
                               warnungen: pd.Series,
                               category: pd.Series,
                               descriptions: pd.Series) -> pd.DataFrame:
    # Prepare each column: fill missing values and convert to string list
    name_ax_list = name_ax.fillna("").astype(str).tolist()
    inhaltsstoffe_list = inhaltsstoffe.fillna("").astype(str).tolist()
    warnungen_list = warnungen.fillna("").astype(str).tolist()
    category_list = category.fillna("").astype(str).tolist()
    descriptions_list = descriptions.fillna("").astype(str).tolist()

    # Get the cached model
    model = get_model()
    
    # Function to compute embeddings for a list of texts using parallel processing
    def encode_texts(text_list):
        num_threads = 4  # adjust based on available cores per executor
        # Split texts into chunks for each thread
        chunk_size = len(text_list) // num_threads + 1
        chunks = [text_list[i:i + chunk_size] for i in range(0, len(text_list), chunk_size)]
        with ThreadPoolExecutor(max_workers=num_threads) as executor:
            # Each thread encodes its chunk concurrently
            results = list(executor.map(
                lambda chunk: model.encode(chunk, batch_size=128, show_progress_bar=False),
                chunks
            ))
        # Flatten the list while preserving order
        embeddings = [emb for sublist in results for emb in sublist]
        # Convert each embedding (a numpy array) to a list
        return [emb.tolist() for emb in embeddings]

    # Compute embeddings for each column
    name_ax_emb = encode_texts(name_ax_list)
    inhaltsstoffe_emb = encode_texts(inhaltsstoffe_list)
    warnungen_emb = encode_texts(warnungen_list)
    category_emb = encode_texts(category_list)
    descriptions_emb = encode_texts(descriptions_list)
    
    # Build a DataFrame with a column for each embedding output
    return pd.DataFrame({
        "Name_AX_embedding": name_ax_emb,
        "Inhaltsstoffe_new_embedding": inhaltsstoffe_emb,
        "Warnungen_new_embedding": warnungen_emb,
        "category_embedding": category_emb,
        "descriptions_embedding": descriptions_emb
    })


In [0]:
df = df.repartition(48)


In [0]:
df_with_embeddings = df.withColumn(
    "embeddings",
    compute_all_embeddings_udf(
        col("Name_AX"),
        col("Inhaltsstoffe_new"),
        col("Warnungen_new"),
        col("category"),
        col("descriptions")
    )
)

# Expand the struct into separate columns and drop the temporary "embeddings" column
for field in embedding_schema.fields:
    df_with_embeddings = df_with_embeddings.withColumn(field.name, col("embeddings." + field.name))
df_with_embeddings = df_with_embeddings.drop("embeddings").cache()

In [0]:
display(df_with_embeddings.limit(10))

In [0]:
import re

def clean_column_name(name):
    return re.sub(r'[ ,;{}()\n\t=]', ' ', name)

for col_name in df_with_embeddings.columns:
    df_with_embeddings = df_with_embeddings.withColumnRenamed(col_name, clean_column_name(col_name))

display(df_with_embeddings.limit(10))

In [0]:
df_with_embeddings.write.format("delta").mode("overwrite").saveAsTable("datascience.at_product_details_individual_embeddings")